In [1]:
import os

import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
from matplotlib.patches import Ellipse
from astropy.visualization import ZScaleInterval

In [2]:
FITS_PATH = "../data/raw/fits/jw02107-o039_t018_miri_f770w_i2d.fits"
CATALOGUE_A_PATH = "../data/raw/catalogue/jwst_bubble_properties_A.txt"
MERGED_PATH = "../data/processed/metadata/mergedCatalogue.txt"

OUTPUT_DIR = "../data/processed/metadata/"
BOUNDS_PATH = os.path.join(OUTPUT_DIR, "curriculum/sizeClassBounds.csv")
MAP_DIR = "../outputs/figures/catalogueMaps"

# ascending in radius; the curriculum consumes them largest-first, but the bin
# edges have to be built in this order
SIZE_CLASSES = ["tiny", "small", "medium", "large", "giants", "edgeLarge"]

SIZE_PERCENTILES = [0.20, 0.40, 0.60, 0.80]
EDGE_PERCENTILE = 0.970

os.makedirs(os.path.dirname(BOUNDS_PATH), exist_ok=True)
os.makedirs(MAP_DIR, exist_ok=True)

In [3]:
# bounds come from catalogue A alone

def build_catalogue_a_radius_bins(catalogue_a):
    radius = catalogue_a["AVG_RAD_PC"]

    percentile_values = radius.quantile(SIZE_PERCENTILES + [EDGE_PERCENTILE])
    q20, q40, q60, q80 = [percentile_values[p] for p in SIZE_PERCENTILES]
    edge_threshold = percentile_values[EDGE_PERCENTILE]

    finite_edges = [q20, q40, q60, q80, edge_threshold]
    if len(set(finite_edges)) != len(finite_edges):
        raise ValueError(
            "Catalogue A radius percentile edges are not unique. "
            "Adjust the binning rule before assigning size classes."
        )

    bin_edges = [float("-inf"), q20, q40, q60, q80, edge_threshold, float("inf")]

    radius_bounds = pd.DataFrame(
        {
            "SIZE_CLASS": SIZE_CLASSES,
            "lower_bound_pc": bin_edges[:-1],
            "upper_bound_pc": bin_edges[1:],
        }
    )

    return bin_edges, radius_bounds


def assign_size_classes(df, bin_edges):
    df = df.copy()

    df["SIZE_CLASS"] = pd.cut(
        df["AVG_RAD_PC"],
        bins=bin_edges,
        labels=SIZE_CLASSES,
        right=False,
    ).astype(str)

    if (df["SIZE_CLASS"] == "nan").any():
        raise ValueError("Some bubbles could not be assigned to a size class.")

    return df

In [4]:
catalogue_a = pd.read_csv(CATALOGUE_A_PATH)
bin_edges, radius_bounds = build_catalogue_a_radius_bins(catalogue_a)

radius_bounds.to_csv(BOUNDS_PATH, index=False)

print("Catalogue A size-class bounds (used by every curriculum stage):")
print(radius_bounds.to_string(index=False))
print("\nSaved:", BOUNDS_PATH)

Catalogue A size-class bounds (used by every curriculum stage):
SIZE_CLASS  lower_bound_pc  upper_bound_pc
      tiny            -inf           18.00
     small           18.00           24.00
    medium           24.00           30.00
     large           30.00           41.00
    giants           41.00          110.21
 edgeLarge          110.21             inf

Saved: ../data/processed/metadata/curriculum/sizeClassBounds.csv


In [5]:
# class width in pixels, against the MIRI F770W resolution

header = fits.open(FITS_PATH)[1].header
pc_per_pixel = (
    ((abs(header["CDELT1"]) + abs(header["CDELT2"])) / 2) * 3600 * 9.77e6 / 206265
)

width = radius_bounds.copy()
width["width_pc"] = width.upper_bound_pc - width.lower_bound_pc
width["width_px"] = width.width_pc / pc_per_pixel

print(f"pc per pixel: {pc_per_pixel:.4f}")
print(f"MIRI F770W PSF FWHM ~0.25\" = {0.25 / (((abs(header['CDELT1']) + abs(header['CDELT2'])) / 2) * 3600):.2f} px\n")
print(width.to_string(index=False, float_format="%.2f"))

pc per pixel: 5.2541
MIRI F770W PSF FWHM ~0.25" = 2.25 px

SIZE_CLASS  lower_bound_pc  upper_bound_pc  width_pc  width_px
      tiny            -inf           18.00       inf       inf
     small           18.00           24.00      6.00      1.14
    medium           24.00           30.00      6.00      1.14
     large           30.00           41.00     11.00      2.09
    giants           41.00          110.21     69.21     13.17
 edgeLarge          110.21             inf       inf       inf


In [6]:
hdu = fits.open(FITS_PATH)
image = hdu[1].data
wcs = WCS(hdu[1].header)

print("Image shape:", image.shape)

df = assign_size_classes(pd.read_csv(MERGED_PATH), bin_edges)

print("Merged catalogue bubbles:", len(df))
print(df["SIZE_CLASS"].value_counts().reindex(SIZE_CLASSES).to_string())

Image shape: (1177, 2380)
Merged catalogue bubbles: 3318
SIZE_CLASS
tiny         378
small        592
medium       667
large        790
giants       778
edgeLarge    113


Set DATE-AVG to '2022-07-17T12:01:53.586' from MJD-AVG.
Set DATE-END to '2022-07-17T12:54:46.016' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -37.754891 from OBSGEO-[XYZ].
Set OBSGEO-H to 1738895745.202 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


In [7]:
def save_stratified_overlay(df, selected_group, catalogue_output_path, vmin, vmax):
    if len(df) == 0:
        print(f"WARNING: No bubbles found for selected_group = {selected_group}")
        return None

    avg_rad_pc_min = df["AVG_RAD_PC"].min()
    avg_rad_pc_max = df["AVG_RAD_PC"].max()

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.imshow(image, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)

    for _, row in df.iterrows():
        bubble_id = int(row["GLOBAL_ID"])

        coord = SkyCoord(row["RA_DMS"], row["DEC_DMS"], frame="icrs")
        x, y = wcs.world_to_pixel(coord)

        semi_major_pix = row["SEMI_MAJ_PC"] / pc_per_pixel
        semi_minor_pix = row["SEMI_MIN_PC"] / pc_per_pixel

        # PA to matplotlib convention
        # theta = -(row["PA_DEG"] - 90)
        theta = (row["PA_DEG"])

        ellipse = Ellipse(
            (x, y),
            width=2 * semi_major_pix,
            height=2 * semi_minor_pix,
            angle=theta,
            edgecolor="red",
            facecolor="none",
            linewidth=2,
        )

        ax.add_patch(ellipse)

        ax.text(x, y, str(bubble_id), color="red", fontsize=5,
                ha="center", va="center")

    ax.set_title(
        f"{selected_group} bubbles (n={len(df)}), "
        f"AVG_RAD_PC {avg_rad_pc_min}-{avg_rad_pc_max} pc"
    )
    ax.set_xlabel("X pixel")
    ax.set_ylabel("Y pixel")

    plt.tight_layout()

    fig.savefig(catalogue_output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print("Saved overlay:", catalogue_output_path)
    return catalogue_output_path

In [8]:
interval = ZScaleInterval()
vmin, vmax = interval.get_limits(image)

for size_class in SIZE_CLASSES:
    subset = df[df["SIZE_CLASS"] == size_class]
    save_stratified_overlay(
        df=subset,
        selected_group=size_class,
        catalogue_output_path=os.path.join(MAP_DIR, f"{size_class}CatalogueMap.png"),
        vmin=vmin,
        vmax=vmax,
    )

Saved overlay: ../outputs/figures/catalogueMaps/tinyCatalogueMap.png


Saved overlay: ../outputs/figures/catalogueMaps/smallCatalogueMap.png


Saved overlay: ../outputs/figures/catalogueMaps/mediumCatalogueMap.png


Saved overlay: ../outputs/figures/catalogueMaps/largeCatalogueMap.png
Saved overlay: ../outputs/figures/catalogueMaps/giantsCatalogueMap.png
Saved overlay: ../outputs/figures/catalogueMaps/edgeLargeCatalogueMap.png
